### Imports and Loading Model

In [1]:
from sentence_transformers import SentenceTransformer
import chromadb

# Setup
model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")

### Indexing Data

In [2]:
chroma = chromadb.Client()

# Reset collections
for name in ["docs", "windows"]:
    try: chroma.delete_collection(name)
    except: pass

# Documents
docs = [
    "The Eiffel Tower is in Paris, France. It is one of the most famous landmarks.",
    "The Colosseum is in Rome, Italy. It is an ancient amphitheater.",
    "The Statue of Liberty is a gift from France, located in New York, USA.",
    "The Burj Khalifa in Dubai is the tallest building in the world."
]

# Parent-doc collection
doc_col = chroma.create_collection("docs")
doc_col.add(
    documents=docs,
    embeddings=model.encode(docs).tolist(),
    ids=[f"d{i}" for i in range(len(docs))]
)

# Window collection (split sentences)
windows = [s for d in docs for s in d.split(". ") if s]
win_col = chroma.create_collection("windows")
win_col.add(
    documents=windows,
    embeddings=model.encode(windows).tolist(),
    ids=[f"w{i}" for i in range(len(windows))]
)

### Basic RAG Query

In [3]:
# Query
query = "Where is the Statue of Liberty?"
q_emb = model.encode([query]).tolist()

### Parent-doc Retrieval

In [4]:
# Parent-doc retrieval
print("Parent-doc results:", doc_col.query(query_embeddings=q_emb, n_results=2)["documents"])

Parent-doc results: [['The Statue of Liberty is a gift from France, located in New York, USA.', 'The Colosseum is in Rome, Italy. It is an ancient amphitheater.']]


### Simple query reformulation

In [5]:
# Simple query reformulation
query_ref = "Statue of Liberty location New York"
q_ref_emb = model.encode([query_ref]).tolist()
print("Reformulated results:", doc_col.query(query_embeddings=q_ref_emb, n_results=2)["documents"])

Reformulated results: [['The Statue of Liberty is a gift from France, located in New York, USA.', 'The Eiffel Tower is in Paris, France. It is one of the most famous landmarks.']]


### Window Retrieval

In [6]:
# Window retrieval
print("Window results:", win_col.query(query_embeddings=q_ref_emb, n_results=2)["documents"])

Window results: [['The Statue of Liberty is a gift from France, located in New York, USA.', 'It is one of the most famous landmarks.']]
